
# Protocolo incremental tipo Bullinaria (2009) en PyTorch

Implementación aproximada con `sklearn.load_digits()`:
- 6 lotes `B1..B6` de **200** muestras (20 por clase).
- Sesiones `T1..T6` entrenando **solo** con el lote actual (sin rehearsal).
- Evaluación al final de cada sesión en: `B_t`, `B_{<t}`, validación y test.
- Opción de **Dual Weights** (pesos rápidos + decay).


In [ ]:

# (Opcional) En Colab suele estar preinstalado.
# !pip install -q torch torchvision scikit-learn matplotlib pandas


## 1) Imports y dispositivo

In [ ]:

import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)
rng = np.random.default_rng(42)


## 2) Lotes B1..B6 y splits de validación/test

In [ ]:

digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)

n_classes = 10
items_per_batch = 200
per_class_per_batch = items_per_batch // n_classes

per_class = {c: np.where(y==c)[0].tolist() for c in range(n_classes)}
for c in per_class: rng.shuffle(per_class[c])

batches = []
cursor = {c: 0 for c in range(n_classes)}
for b in range(6):
    idx = []
    for c in range(n_classes):
        start, end = cursor[c], cursor[c] + per_class_per_batch
        idx.extend(per_class[c][start:end])
        cursor[c] = end
    rng.shuffle(idx)
    batches.append(idx)

used = set(sum(batches, []))
rest_idx = [i for i in range(len(X)) if i not in used]
rng.shuffle(rest_idx)
split = int(0.6*len(rest_idx))
val_idx, test_idx = rest_idx[:split], rest_idx[split:]

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

def make_loader(idx, bs=128, shuffle=False):
    ds = TensorDataset(X_tensor[idx], y_tensor[idx])
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

batch_loaders = [make_loader(idx, bs=64, shuffle=True) for idx in batches]
val_loader   = make_loader(val_idx, shuffle=False)
test_loader  = make_loader(test_idx, shuffle=False)

print("Tamaños por lote:", [len(b) for b in batches], "| Val:", len(val_idx), "| Test:", len(test_idx))


## 3) Modelos: MLP y Dual Weights

In [ ]:

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

class DualLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.w_main = nn.Parameter(torch.empty(out_f, in_f))
        self.b_main = nn.Parameter(torch.empty(out_f))
        self.w_fast = nn.Parameter(torch.zeros(out_f, in_f))
        self.b_fast = nn.Parameter(torch.zeros(out_f))
        nn.init.kaiming_uniform_(self.w_main, a=np.sqrt(5))
        bound = 1.0/np.sqrt(in_f)
        nn.init.uniform_(self.b_main, -bound, bound)
    def forward(self, x):
        W = self.w_main + self.w_fast
        b = self.b_main + self.b_fast
        return x @ W.T + b

class MLPDual(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = DualLinear(64, 128)
        self.fc2 = DualLinear(128, 64)
        self.fc3 = DualLinear(64, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


## 4) Entrenamiento y evaluación

In [ ]:

criterion = nn.CrossEntropyLoss()

def train_epochs(model, loader, optimizer, epochs, device):
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

@torch.no_grad()
def accuracy(model, loader, device):
    model.eval()
    total, correct = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    return correct/total if total else float('nan')


## 5) Protocolo T1..T6

In [ ]:

use_dual = True
epochs_per_session = 200

model = (MLPDual() if use_dual else MLP()).to(device)

if use_dual:
    main_params = [p for n,p in model.named_parameters() if 'main' in n]
    fast_params = [p for n,p in model.named_parameters() if 'fast' in n]
    optimizer = torch.optim.SGD([
        {'params': main_params, 'lr': 1e-3, 'weight_decay': 0.0},
        {'params': fast_params, 'lr': 1e-1, 'weight_decay': 1e-3},
    ], momentum=0.9)
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

rows = []
for t in range(6):
    train_epochs(model, batch_loaders[t], optimizer, epochs_per_session, device)
    acc_bt = accuracy(model, batch_loaders[t], device)
    if t > 0:
        import numpy as np
        prev_idx = np.concatenate(batches[:t]).tolist()
        prev_loader = DataLoader(TensorDataset(X_tensor[prev_idx], y_tensor[prev_idx]),
                                 batch_size=128, shuffle=False)
        acc_prev = accuracy(model, prev_loader, device)
    else:
        acc_prev = float('nan')
    acc_val = accuracy(model, val_loader, device)
    acc_test = accuracy(model, test_loader, device)
    rows.append({'T': t+1, 'Acc_Bt': acc_bt, 'Acc_prev': acc_prev,
                 'Acc_val': acc_val, 'Acc_test': acc_test,
                 'Model': 'DualWeights' if use_dual else 'MLP'})
results_df = pd.DataFrame(rows)
results_df


## 6) Guardado y gráficas

In [ ]:

csv_path = "bullinaria_protocol_results.csv"
results_df.to_csv(csv_path, index=False)
print("CSV guardado en:", csv_path)

plt.figure()
plt.plot(results_df['T'], results_df['Acc_Bt'], marker='o', label='Acc_Bt (actual)')
plt.plot(results_df['T'], results_df['Acc_prev'], marker='o', label='Acc_prev (anteriores)')
plt.plot(results_df['T'], results_df['Acc_val'], marker='o', label='Acc_val')
plt.plot(results_df['T'], results_df['Acc_test'], marker='o', label='Acc_test')
plt.xlabel('Sesión T'); plt.ylabel('Accuracy')
plt.title(f"Evolución incremental ({results_df['Model'][0]})")
plt.legend(); plt.show()

results_df
